# 04 · Preprocessing

Build the preprocessing pipeline that **every** downstream notebook (`05_feature_engineering`, `06_ml_prediction`) will reuse. Three rules:

1. **Fit on TRAIN ONLY.** Validation and test see the same transformer with parameters frozen at training time. No leakage.
2. **V1–V28 pass through.** They are already PCA-scaled; rescaling them adds nothing and only invites bugs.
3. **Don't undersample.** We handle class imbalance with **class weights** (`class_weight='balanced'` for logreg, `scale_pos_weight=neg/pos` for XGBoost). Undersampling throws away ~99.8% of legitimate data and, if done before the split, leaks distributional information.

Output of `build_preprocessor()`: 28 passthrough V-columns + `amount_log` + `amount_robust` + `hour_sin` + `hour_cos` = **32 columns**.

In [ ]:
# project-root bootstrap — portable across VS Code / Jupyter / CLI
import os
from pathlib import Path
_p = Path.cwd()
while not (_p / 'config' / 'config.yaml').exists() and _p != _p.parent:
    _p = _p.parent
os.chdir(_p)
print('working dir:', Path.cwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fraud.io import read_parquet
from fraud.preprocess import build_preprocessor, V_COLS

plt.rcParams.update({'figure.facecolor':'white','axes.spines.top':False,
                     'axes.spines.right':False,'axes.titleweight':'bold'})
FIG = Path('reports/figures'); FIG.mkdir(parents=True, exist_ok=True)

FEATURES = ['Time', 'Amount'] + V_COLS
train = read_parquet('data/processed/train.parquet')
valid = read_parquet('data/processed/valid.parquet')
test  = read_parquet('data/processed/test.parquet')
{'train': train.shape, 'valid': valid.shape, 'test': test.shape}

## 1 · Build and fit (on train only)

`build_preprocessor()` returns a sklearn `ColumnTransformer` that we fit on train and reuse on valid and test. The fitted object is what `06_ml_prediction` will pickle alongside the model.

In [ ]:
pre = build_preprocessor()
X_train = pre.fit_transform(train[FEATURES])
X_valid = pre.transform(valid[FEATURES])
X_test  = pre.transform(test[FEATURES])
print(f"transformed shapes  train={X_train.shape}  valid={X_valid.shape}  test={X_test.shape}")
print('expected: 32 cols = 28 V + amount_log + amount_robust + hour_sin + hour_cos')

## 2 · Inspect the output — what each transformer produces

Sanity check: the first 28 columns should be untouched copies of `V1..V28`, and the last 4 should be the engineered ones.

In [ ]:
COLS = V_COLS + ['amount_log', 'amount_robust', 'hour_sin', 'hour_cos']
X_train_df = pd.DataFrame(X_train, columns=COLS)

passthrough_ok = np.allclose(X_train_df[V_COLS].values, train[V_COLS].values)
print(f"V1..V28 passthrough preserves values exactly: {passthrough_ok}")

X_train_df[['amount_log','amount_robust','hour_sin','hour_cos']].head()

## 3 · No-leakage check — Amount distribution after robust scaling

If the fit-on-train-only contract is respected, the scaled validation distribution should look **similar but not identical** to the training one (same shape, different sample). If we had accidentally re-fit on valid, the two would line up too perfectly. The before/after panels make the transform itself visible too.

In [ ]:
X_valid_df = pd.DataFrame(X_valid, columns=COLS)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(train['Amount'].clip(upper=2000), bins=60, alpha=.7, color='#3498db')
axes[0].set(title='Amount (raw)', xlabel='$', ylabel='count')

axes[1].hist(X_train_df['amount_log'], bins=60, alpha=.7, color='#27ae60')
axes[1].set(title='amount_log = log1p(Amount)', xlabel='log1p $')

axes[2].hist(X_train_df['amount_robust'].clip(-2, 10), bins=60,
             alpha=.55, color='#3498db', label='train (fit)', density=True)
axes[2].hist(X_valid_df['amount_robust'].clip(-2, 10), bins=60,
             alpha=.55, color='#e67e22', label='valid (transform only)', density=True)
axes[2].set(title='amount_robust — train vs valid', xlabel='robust-scaled Amount')
axes[2].legend()

fig.suptitle('Preprocessing — Amount transforms (no-leakage check)', fontweight='bold')
fig.tight_layout()
fig.savefig(FIG/'preprocessing_amount.png', dpi=120, bbox_inches='tight')
plt.show()

## 4 · Imbalance strategy — class weights, not undersampling

**Why not undersample globally?** It throws away ~99.8% of legitimate data, and if done before the train/valid/test split it leaks information about the test distribution. Class weights keep all the data and let the model see the true prior.

- For **logistic regression** → `class_weight='balanced'` (sklearn computes per-class weight inversely proportional to frequency).
- For **XGBoost** → `scale_pos_weight = neg / pos` (~577 on this training set).

Both numbers flow straight into `06_ml_prediction` without modification.

In [ ]:
y_train = train['Class'].values
neg, pos = int((y_train == 0).sum()), int((y_train == 1).sum())
spw = neg / max(pos, 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].barh(['legit (0)','fraud (1)'], [neg, pos], color=['#3498db','#c0392b'])
axes[0].set_xscale('log')
axes[0].set(title='Training class counts (log scale)', xlabel='count (log)')
for i, v in enumerate([neg, pos]):
    axes[0].text(v, i, f' {v:,}', va='center', fontsize=10)

axes[1].axis('off')
axes[1].text(0.05, 0.85, 'Imbalance summary', fontsize=12, fontweight='bold', color='#0a3d62')
axes[1].text(0.05, 0.65, f'negatives = {neg:,}',       fontsize=11)
axes[1].text(0.05, 0.50, f'positives = {pos:,}',       fontsize=11)
axes[1].text(0.05, 0.35, f'ratio     = {spw:,.1f} : 1', fontsize=11)
axes[1].text(0.05, 0.15, f'scale_pos_weight (XGB) = {spw:,.2f}',
             fontsize=11, color='#c0392b', fontweight='bold')
fig.suptitle('Preprocessing — class imbalance', fontweight='bold')
fig.tight_layout()
fig.savefig(FIG/'preprocessing_imbalance.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'scale_pos_weight = {spw:.2f}')

**Read:**

- `build_preprocessor()` produces a fitted, picklable transformer with 32 output columns; V1–V28 pass through untouched; `Amount` becomes `(log, robust)`; `Time` becomes `(hour_sin, hour_cos)`.
- Train was the only split used to fit; valid and test were transform-only.
- Class imbalance is ~577 : 1; the `scale_pos_weight` and `class_weight='balanced'` settings carry into `05` and `06` directly.

**Next:** `05_feature_engineering` tests whether the four engineered columns above earn their keep on both a linear and a tree model. Then `06_ml_prediction` trains, tunes, calibrates, and picks a business-cost-optimal threshold.